# `nn.Module`、参数与 logits

## 学习目标

能够定义模块、检查参数和 buffer 的注册状态、追踪前向传播的张量形状，并正确理解分类 logits、训练/评估模式与 forward hook。

## 概念模型与执行路径

`nn.Module` 不只是一个可调用对象，它还管理子模块、可训练参数、持久化状态、设备移动和训练模式。调用 `model(x)` 时，`Module.__call__` 会处理 hooks 等框架逻辑，再进入用户定义的 `forward`。分类模型通常输出未归一化的 logits，交叉熵内部会以数值稳定的方式完成 `log_softmax` 和负对数似然计算。

本页的数据流是：`(batch, 4)` 输入 → 隐藏线性层 → ReLU → 分类线性层 → `(batch, 3)` logits → 交叉熵标量损失。

In [1]:
import torch
from torch import nn
class ContainerDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(4, 4), nn.ReLU()])
        self.lookup = nn.ModuleDict({'head': nn.Linear(4, 2)})
        self.register_buffer('scale', torch.ones(1))
    def forward(self, x):
        return self.lookup['head'](self.layers[1](self.layers[0](x))) * self.scale

demo = ContainerDemo()
assert 'scale' in demo.state_dict() and any('layers.0' in n for n, _ in demo.named_parameters())
dropout = nn.Dropout(0.5)
dropout.train(); train_output = dropout(torch.ones(100))
dropout.eval(); eval_output = dropout(torch.ones(100))
assert not torch.equal(train_output, eval_output)
print('registered parameters and buffer:', list(demo.state_dict()))

registered parameters and buffer: ['scale', 'layers.0.weight', 'layers.0.bias', 'lookup.head.weight', 'lookup.head.bias']


## 实验 0：容器、buffer 和训练模式

**实验目的**：验证 `nn.Module` 如何追踪不同种类的状态，以及 `train()`/`eval()` 如何改变具有模式依赖行为的层。

`ModuleList` 和 `ModuleDict` 的使用方式类似 Python 容器，但放入其中的层会被注册为子模块。于是 `layers.0.weight`、`lookup.head.bias` 等参数会出现在 `named_parameters()` 和 `state_dict()` 中，也会随 `demo.to(device)` 一起移动。普通 Python `list`/`dict` 不提供这种注册能力。

`scale` 通过 `register_buffer` 注册。buffer 是模型状态的一部分，会进入 `state_dict()` 并随模型移动设备，但不是 `Parameter`，因此优化器不会更新它。BatchNorm 的运行均值、运行方差就是典型 buffer。若某个 buffer 不应保存进 checkpoint，可以设置 `persistent=False`。

Dropout 在训练模式下以概率 0.5 随机置零，并放大保留值以维持期望；在评估模式下等价于恒等映射。因此两个输出通常不同。这里的断言具有极低概率的随机偶然性：100 个元素恰好都未被置零时仍因训练输出缩放而不同；要做完全确定的测试，可固定随机种子并检查评估输出等于全 1。

**观察重点**：`train()` 和 `eval()` 递归设置模块的 `training` 标志，但不会开启或关闭梯度。推理时通常同时使用 `model.eval()` 与 `torch.no_grad()`/`torch.inference_mode()`，两者职责不同。

### 实验 1：用 `nn.Module` 定义并注册网络结构

**实验目的**：构建一个最小的多层感知机，并通过打印模型确认子模块已经注册。`Classifier` 继承 `nn.Module`，必须先调用 `super().__init__()`，框架才能建立用于登记参数、buffer 和子模块的内部容器。

`nn.Sequential` 按声明顺序连接三层：`Linear(4, 8)` 把每个样本的 4 个输入特征映射为 8 个隐藏特征，`ReLU` 引入非线性，`Linear(8, 3)` 输出 3 个类别的分数。若移除 ReLU，两次线性变换整体仍等价于一次线性变换，增加隐藏层便无法提升模型表示非线性关系的能力。

调用 `model(batch)` 而不是直接调用 `model.forward(batch)`，因为前者会经过 `nn.Module.__call__`，从而正确执行 forward hooks、混合精度和框架的其他包装逻辑。

**预期现象**：打印结果展示 `net` 及其编号为 0、1、2 的子模块；模型尚未接收数据，但参数已经在各个 `Linear` 层初始化并注册。

In [2]:
import torch
from torch import nn

class Classifier(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8, classes=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, classes))

    def forward(self, inputs):
        return self.net(inputs)

model = Classifier()
print(model)


Classifier(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=3, bias=True)
  )
)


### 实验 2：追踪前向形状与参数注册

**实验目的**：验证 batch 维在网络中保持不变，并检查每个可训练参数的名称、形状和求导状态。输入 `batch` 的形状是 `(5, 4)`：5 是样本数，4 必须匹配第一层的 `in_features`。最终 logits 为 `(5, 3)`，表示每个样本对应 3 个类别分数。

PyTorch 的 `Linear(in_features, out_features)` 将权重保存为 `(out_features, in_features)`。前向计算等价于 $y=xW^T+b$，所以两层参数形状依次为：

- `net.0.weight`: `(8, 4)`，`net.0.bias`: `(8,)`；
- `net.2.weight`: `(3, 8)`，`net.2.bias`: `(3,)`。

参数总量为 $8\times4+8+3\times8+3=67$。`named_parameters()` 只遍历已注册的 `Parameter`；所有结果的 `requires_grad=True`，意味着后续损失反向传播会为它们计算梯度。

**观察重点**：logits 的第二维必须和类别数一致。若输入最后一维不是 4，矩阵乘法会因形状不兼容而报错。

In [3]:
batch = torch.randn(5, 4)
logits = model(batch)
print("input -> logits:", batch.shape, "->", logits.shape)
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape), parameter.requires_grad)


input -> logits: torch.Size([5, 4]) -> torch.Size([5, 3])
net.0.weight (8, 4) True
net.0.bias (8,) True
net.2.weight (3, 8) True
net.2.bias (3,) True


### 实验 3：从 logits 计算分类损失与概率

**实验目的**：理解训练使用的 logits、标签和用于展示的概率之间的关系。`labels` 的形状为 `(5,)`，每个整数是对应样本的正确类别索引，取值范围必须是 `[0, 3)`，dtype 通常为 `torch.long`。

`CrossEntropyLoss` 接收原始 logits，不应先调用 softmax。它对每个样本计算 $-\log(\operatorname{softmax}(z)_{y})$，默认再对 batch 求平均；其内部融合了 `log_softmax` 与 NLLLoss，数值上比“先 softmax、再取 log”更稳定。这里的 loss 取决于随机初始化和随机输入，因此每次新启动内核运行可能不同。

`softmax(dim=1)` 仅用于把每行 3 个 logits 转成非负、和约为 1 的概率。必须沿类别维 `dim=1` 归一化，而不是沿 batch 维。浮点舍入可能让打印值非常接近但不精确等于 1。

**反向路径**：若随后调用 `loss.backward()`，梯度会从标量损失经过分类层、ReLU 和隐藏层，累积到实验 2 列出的全部参数 `.grad` 中。

In [4]:
labels = torch.tensor([0, 2, 1, 0, 2])
loss = nn.CrossEntropyLoss()(logits, labels)
probabilities = logits.softmax(dim=1)
print("loss:", loss.item())
print("probability row sums:", probabilities.sum(dim=1))


loss: 1.1168904304504395
probability row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


### 实验 4：使用 forward hook 观察中间激活

**实验目的**：在不修改 `forward` 返回值的前提下，捕获第一层线性变换的输出。`register_forward_hook` 注册的回调会在目标模块完成前向计算后收到三个对象：模块本身、输入参数元组 `args` 和输出 `output`。

hook 挂在 `model.net[0]`，因此捕获的是 ReLU **之前**的线性输出，形状为 `(5, 8)`。其中可以包含负值；若把 hook 改挂到 `model.net[1]`，捕获的 ReLU 输出将保持相同形状，但所有值都非负。

保存时调用 `output.detach()`，避免调试用的 `activations` 字典长期引用计算图，造成不必要的内存占用。`handle.remove()` 同样重要：若重复注册却不移除，同一 hook 会在后续每次前向传播中持续执行。

**适用场景**：检查形状、可视化激活和诊断数值异常。hook 会增加运行开销并引入隐藏副作用，通常只在调试或特征采集阶段启用。

In [5]:
activations = {}
handle = model.net[0].register_forward_hook(lambda module, args, output: activations.update(linear=output.detach()))
_ = model(batch)
handle.remove()
print("captured hidden shape:", activations["linear"].shape)


captured hidden shape: torch.Size([5, 8])


## 底层机制

当一个 `Module` 或 `Parameter` 通过属性赋值，或被放入 `Sequential`、`ModuleList`、`ModuleDict` 等专用容器时，`nn.Module.__setattr__` 会把它登记到内部模块/参数映射中。`state_dict()` 随后递归收集参数和持久化 buffer，优化器则通过 `model.parameters()` 取得需要更新的参数。

把层放进普通 Python list 会绕过注册：层虽然仍可被手动调用，但其参数不会出现在 `model.parameters()` 中，也不会自动随 `model.to(device)` 移动或写入 checkpoint。另一个常见陷阱是在 `forward` 中临时创建 `nn.Linear`：它会在每次调用时重新随机初始化，并且优化器从未持有其参数。带状态的层应在 `__init__` 中创建并注册。

## 官方教程补充

**对应官方源文件：** `beginner_source/basics/buildmodel_tutorial.py`、`beginner_source/blitz/neural_networks_tutorial.py`、`recipes_source/recipes/defining_a_neural_network.py`

官方示例的核心契约是：子模块和 `nn.Parameter` 只要作为属性赋值，就会被注册到 `state_dict`，并被 `.to()`、`.train()` 和优化器统一管理。`forward` 只描述数据流，调用 `model(x)` 还会经过 Module 的 hook 等机制，因此不要直接调用 `model.forward(x)`。分类头返回 logits，让交叉熵内部处理 LogSoftmax。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

不运行代码，先回答再验证：

1. 该模型共有多少个可训练标量参数？分别来自哪里？
2. 第一层权重为什么保存为 `(8, 4)`，而输入是 `(5, 4)`？写出对应矩阵乘法。
3. `model.eval()` 后，各参数的 `requires_grad` 会不会变成 `False`？为什么？
4. 为什么 `CrossEntropyLoss` 前不应手动 softmax？
5. 实验 4 捕获的张量为何可能包含负值？怎样捕获 ReLU 之后的结果？

## 试一试

1. **扩展网络**：在分类层前增加 `Linear(8, 6)` 和 `ReLU()`，用 hook 记录每个子层的输出形状，确认最终 logits 仍为 `(5, 3)`，并重新计算参数量。
2. **验证模式边界**：分别在 `dropout.train()` 和 `dropout.eval()` 下重复运行同一输入；再检查 `torch.is_grad_enabled()`，说明训练/评估模式与梯度开关为何是两个独立维度。
3. **验证注册机制**：把两个 `Linear` 分别放进普通 `list` 和 `ModuleList`，比较 `list(model.named_parameters())`、`state_dict()` 以及设备移动后的参数设备。
4. **执行一次训练步骤**：创建优化器，依次执行清梯度、前向、计算损失、反向和更新；比较更新前后的第一层权重，确认梯度确实驱动了参数变化。

## 常见错误与调试

- **先 softmax 再传给交叉熵**：输入语义错误且数值稳定性下降。直接传原始 logits。
- **在 `forward` 中临时创建带参数层**：每次前向都会重新初始化，参数也不会被既有优化器管理。把层移到 `__init__`。
- **忘记调用 `super().__init__()`**：子模块或参数赋值可能直接报错，因为注册容器尚未初始化。
- **标签 dtype 或形状错误**：类别索引模式下应使用形如 `(batch,)` 的 `torch.long` 标签，并确保索引未超出类别范围。
- **误以为 `eval()` 会禁用梯度**：它只切换 Dropout/BatchNorm 等层的行为。推理还需使用 `torch.no_grad()` 或 `torch.inference_mode()`。
- **普通 list 中的层未注册**：表现为优化器参数缺失、保存加载遗漏或 CPU/GPU 设备不一致。使用 `ModuleList`、`ModuleDict` 或 `Sequential`。
- **hook 重复触发或占用内存**：保存 `handle` 并及时 `remove()`；只观察数值时对输出调用 `detach()`。